In [1]:
#import pandas as pd
#import os
from create_model_input_data_format_functions import *

In [2]:
#to define later values for hydrogen if not provided (all 0 but necessary). 
investment = False
hydrogen_investment=False

In [3]:
#set szenario
szenario_year = '2019'
szenario_name = 'run_' + szenario_year

In [4]:
#output specifications
output_file_path = os.path.join('..', '..','01_data', '01_input_data', '02_processed', '01_paper_IAEE')
output_file_name = '\inputs_IAEE_2025_' + szenario_name + '.xlsx'
output_file_name_edges_raw = '\inputs_IAEE_2025_edges_raw.xlsx'
#create full ouput paths
output_file_path_excel  = output_file_path + output_file_name
full_output_file_path = os.path.abspath(os.path.join(os.getcwd(), output_file_path_excel))
output_file_path_edges_raw  = output_file_path + output_file_name_edges_raw
full_edges_raw_file_path = os.path.abspath(os.path.join(os.getcwd(), output_file_path_edges_raw))

## Import Data

In [23]:
# Specify the path to your Excel file
input_file_path_1 = os.path.join('..', '..','01_data', '01_input_data', '02_processed')
excel_file_name = '\\Gas_Import_Russian_Invasion.xlsx'
LNG_cost_file_name = '\\LNG_Transportation_Cost_Calculation.xlsx'
# Specify the path to your Excel file
input_file_path_2 = os.path.join('..', '..','00_code_base', '07_input_data_preparation')
distances_file_name = '\\distances_' + szenario_year + '.xlsx'
# Specify the path to your Excel file
input_file_path_3 = os.path.join('..', '..','01_data', '01_input_data', '01_raw', '01_Russian_War_Case')
szenarios_update_file_name = '\\Szenarios_update_information.xlsx'

#create full input paths
input_file_path_excel  = input_file_path_1 + excel_file_name
full_input_path_1 = os.path.abspath(os.path.join(os.getcwd(), input_file_path_excel))
input_file_path_LNG = input_file_path_1 + LNG_cost_file_name
full_input_path_LNG_cost = os.path.abspath(os.path.join(os.getcwd(), input_file_path_LNG))
input_file_path_2  = input_file_path_2 + distances_file_name
full_input_path_2 = os.path.abspath(os.path.join(os.getcwd(), input_file_path_2))
full_input_path_3 = os.path.abspath(os.path.join(os.getcwd(), input_file_path_3 + szenarios_update_file_name))

In [24]:
df_LNG_global = pd.read_excel(full_input_path_1, sheet_name='global_LNG_connections_' + szenario_year)
df_LNG_europe = pd.read_excel(full_input_path_1, sheet_name='nodes_lng_europe_' + szenario_year)
df_production_europe = pd.read_excel(full_input_path_1, sheet_name='nodes_gas_prod_europe_' + szenario_year)# + '_SP')
df_consumption_europe = pd.read_excel(full_input_path_1, sheet_name='nodes_demand_europe_' + szenario_year)# + '_SP')
df_production_global = pd.read_excel(full_input_path_1, sheet_name='nodes_gas_prod_world_' + szenario_year)
df_consumption_global = pd.read_excel(full_input_path_1, sheet_name='nodes_demand_world_' + szenario_year)
df_pipelines_europe = pd.read_excel(full_input_path_1, sheet_name='Connections_' + szenario_year)
df_pipelines_global = pd.read_excel(full_input_path_1, sheet_name='Global_Connections')

df_updates = pd.read_excel(full_input_path_3, sheet_name = szenario_year )#+ '_NOR')
#only relevant for the investment case
if investment == True:
    df_updates_invest = pd.read_excel(full_input_path_3, sheet_name = szenario_year + '_InvesPipes')

#load LNG and production cost data 
df_LNG_cost = pd.read_excel(full_input_path_LNG_cost, sheet_name='LNG_Cost_Params')
df_production_cost = pd.read_excel(full_input_path_LNG_cost, sheet_name='Production_Cost_Nodes')
#to be alligned in the future
production_cost_df = df_production_cost

#load input for inner-European distances
df_distances = pd.read_excel(full_input_path_2, sheet_name='distances')
#Drop the "Unnamed: 0" column in df_distances
df_distances.drop(columns=["Unnamed: 0"], inplace=True)

In [25]:
df_updates

,Commodity,Source,Destination,costs_new
0,Hydrogen,RU,DE,0


In [26]:
#adjust the data frames and remove unnecessary content
#gobal demand
df_consumption_global = df_consumption_global.iloc[:-3]
#gobal production
df_production_global = df_production_global.iloc[:-4]
#globale pipelines
df_pipelines_global = df_pipelines_global.iloc[:, :-5]
#europe demand
df_consumption_europe = df_consumption_europe.iloc[:-11]
#europe production
df_production_europe = df_production_europe.iloc[:-7]
df_production_europe = df_production_europe.iloc[:, :-7]
#europe LNG
df_LNG_europe = df_LNG_europe.iloc[:, :-6]
#add a From column to the European LNG data
df_LNG_europe.insert(1, 'From', df_LNG_europe['To'] + '_LNG')
#LNG global
df_LNG_global = df_LNG_global.iloc[:, :-5]

#adjust naming
df_distances = df_distances.rename(columns={"from [NUTS_ID]": "From"})
df_distances = df_distances.rename(columns={"to [NUTS_ID]": "To"})

In [27]:
#to remove if in the df as not part of the gas network (no data for interconnectors) 
regions_to_remove_list = ['Other Europe', 'San Marino', 'Malta', 'Kosovo', 'Zyprus', 'Montenegro']
df_consumption_europe = remove_rows_containing_strings(df_consumption_europe, regions_to_remove_list)
df_production_europe = remove_rows_containing_strings(df_production_europe, regions_to_remove_list)
df_LNG_europe = remove_rows_containing_strings(df_LNG_europe, regions_to_remove_list)

In [21]:
#set values for parameters
cost_factor_transportation = df_LNG_cost.loc[df_LNG_cost['type'] == 'cost per GWh and km', 'Cost'].values[0]
cost_factor_liquefaction = df_LNG_cost.loc[df_LNG_cost['type'] == 'liquefication per GWh', 'Cost'].values[0]
cost_factor_regasification = df_LNG_cost.loc[df_LNG_cost['type'] == 'regasification per GWh', 'Cost'].values[0]
cost_factor_panama = df_LNG_cost.loc[df_LNG_cost['type'] == 'Cost Panama per GWh', 'Cost'].values[0]
cost_factor_suez = df_LNG_cost.loc[df_LNG_cost['type'] == 'Cost Suez per GWh', 'Cost'].values[0]

cost_factor_pipelines = df_LNG_cost.loc[df_LNG_cost['type'] == 'cost pipe per GWh and km', 'Cost'].values[0]

### Demand and Supply input sheet

In [22]:
df_consumption_europe

,Country,Long_name,Mrd m3 [2019],TWh [2019],Population 2019 [Tsd],TWh by share,GWh [2019]
0,AL,Albania,NaN,0.000000,2862.400,-7.737910,-7737.910080
1,AT,Austria,8.900441,-86.957309,8858.800,0.000000,-86957.308932
2,BE,Belgium,17.386897,-169.869989,11455.500,0.000000,-169869.988575
3,BA,Bosnia and Herzegovina,NaN,0.000000,3492.000,-9.439904,-5110.000000
4,BG,Bulgaria,2.830875,-27.657649,7000.000,0.000000,-27657.648750
5,HR,Croatia,2.798142,-27.337850,4076.200,0.000000,-27337.849511
6,CZ,Czech Republic,8.324950,-81.334761,10649.800,0.000000,-81334.761500
7,DK,Denmark,2.897778,-28.311290,5806.100,0.000000,-28311.289548
8,EE,Estonia,0.466055,-4.553361,1324.800,0.000000,-4553.361149
9,FI,Finland,2.033833,-19.870552,5517.900,0.000000,-19870.551667


In [14]:
#demand Europe
df_demand_europe = create_supply_demand_df(df_consumption_europe, supply=False)
#supply Europe
df_supply_europe = create_supply_demand_df(df_production_europe)

KeyError: 'GWh [2020]'

In [ ]:
#demand Europe
df_demand_global = create_supply_demand_df(df_consumption_global, supply=False)
#supply global
df_supply_global = create_supply_demand_df(df_production_global)

In [ ]:
# Combine the demand and supply data frames
df_supply_demand = pd.concat([df_demand_europe, df_supply_europe, df_demand_global, df_supply_global], ignore_index=True)

In [45]:
# Apply the function to each row and create a new column 'Cost per km'
df_LNG_global['Cost'] = calculate_cost_vectorized(
    df_LNG_global,
    cost_factor_transportation=cost_factor_transportation,
    cost_factor_suez=cost_factor_suez,
    cost_factor_panama=cost_factor_panama
)

# Create a new dataframe with the relevant columns
df_cost_per_km = df_LNG_global[['From', 'To', 'Distance [km]', 'Suez or Panama', 'Cost']]

In [46]:
#add export and import information
df_LNG_global['From'] = df_LNG_global['From'].str.replace('_LNG', '_LNG_exp')
df_LNG_global['To'] = df_LNG_global['To'].str.replace('_LNG', '_LNG_imp')

In [47]:
# Get LNG liquefaction cost df
LNG_liquefaction_df = create_LNG_liquefaction_df(df_LNG_global, cost_factor_liquefaction)
# Get LNG regasification cost df
LNG_regasification_df = create_LNG_regasification_df(df_LNG_global, cost_factor_regasification)

In [48]:
#add regasification cost to European LNG import nodes
df_LNG_europe.insert(3, "Cost", cost_factor_regasification)
#add export and import information
df_LNG_europe['From'] = df_LNG_europe['From'].str.replace('_LNG', '_LNG_imp')
df_LNG_europe['To'] = df_LNG_europe['To'].str.replace('_LNG', '_LNG_exp')

In [49]:
#Pipeline cost
#calculate European pipeline cost
df_european_pipe_transport_cost = pipeline_transport_cost(df_distances, cost_factor_pipelines)
df_global_pipe_transport_cost = pipeline_transport_cost(df_pipelines_global, cost_factor_pipelines)

In [50]:
# Example usage:
df_pipelines_europe = integrate_pipeline_costs(df_pipelines_europe, df_european_pipe_transport_cost)

In [51]:
df_europe_pipe_edges = process_european_pipeline_edges(df_pipelines_europe)

In [52]:
df_global_pipe_edges = process_global_pipeline_edges(df_global_pipe_transport_cost)
df_liquefaction_LNG_edges = process_LNG_liquefaction_edges(LNG_liquefaction_df)
df_regasification_LNG_edges = process_LNG_regasification_edges(LNG_regasification_df)
df_europe_LNG_edges = process_LNG_import_edges(df_LNG_europe)
df_europe_pipe_edges = process_european_pipeline_edges(df_pipelines_europe)
df_LNG_global_edges = process_LNG_global_edges(df_LNG_global)
#reomve duplicates from the regasification edges df
df_regasification_LNG_edges = remove_existing_LNG_edges(df_regasification_LNG_edges, df_europe_LNG_edges)
#add missing edges for the production
df_production_edges = create_Production_node_edges(df_supply_demand, production_cost_df)
#add missing edges for LNG exporting countries
df_missing_LNG_export_edges = ensure_direct_connections(df_LNG_global_edges, df_LNG_europe, cost_factor_liquefaction)

In [53]:
# Apply the create_edges_cap_cost_dataframe function to get edges input 
df_edges_cap_cost = merge_all_edges(
    df_europe_pipe_edges, 
    df_europe_LNG_edges, 
    df_regasification_LNG_edges, 
    df_liquefaction_LNG_edges, 
    df_global_pipe_edges, 
    df_LNG_global_edges,
    df_missing_LNG_export_edges,
    df_production_edges
)

In [54]:
df_edges_cap_cost

,Commodity,Source,Destination,initial_capacities,max_capacities,costs_edge,new_build_cost,conversion_cost,conversion_capacity_factor
0,Methane,PL,DK,33288.0,33288.0,1279.087442,1000000,0,1
1,Methane,LT,RU,0.0,0.0,6330.280756,1000000,0,1
2,Methane,ES,PT,52560.0,52560.0,655.821896,1000000,0,1
3,Methane,RU,FI,0.0,0.0,5049.665541,1000000,0,1
4,Methane,RO,UA,44603.0,44603.0,1029.314500,1000000,0,1
...,...,...,...,...,...,...,...,...,...
449,Methane,IN_Prod,IN,999999999.0,999999999.0,32674.666811,1000000,0,1
450,Methane,AS_Prod,AS,999999999.0,999999999.0,32674.666811,1000000,0,1
451,Methane,CR_Prod,CR,999999999.0,999999999.0,19449.206435,1000000,0,1
452,Methane,EG_Prod,EG,999999999.0,999999999.0,17504.285792,1000000,0,1


In [55]:
df_LNG_global_edges

,Commodity,Source,Destination,initial_capacities,max_capacities,costs_edge,new_build_cost,conversion_cost,conversion_capacity_factor
0,Methane,AF_LNG_exp,NL_LNG_imp,999999999.0,999999999.0,571.819800,1000000,0,1
1,Methane,RU_LNG_exp,LT_LNG_imp,999999999.0,999999999.0,394.729254,1000000,0,1
2,Methane,EG_LNG_exp,IT_LNG_imp,999999999.0,999999999.0,160.508548,1000000,0,1
3,Methane,EG_LNG_exp,HR_LNG_imp,999999999.0,999999999.0,162.969938,1000000,0,1
4,Methane,AU_LNG_exp,FR_LNG_imp,999999999.0,999999999.0,1069.351224,1000000,0,1
...,...,...,...,...,...,...,...,...,...
233,Methane,USA_LNG_exp,DE_LNG_imp,999999999.0,999999999.0,675.586827,1000000,0,1
234,Methane,AF_LNG_exp,UK_LNG_imp,999999999.0,999999999.0,558.217380,1000000,0,1
235,Methane,RU_LNG_exp,CN_LNG_imp,999999999.0,999999999.0,1717.706216,1000000,0,1
236,Methane,QA_LNG_exp,NL_LNG_imp,999999999.0,999999999.0,822.197335,1000000,0,1


### check for hydrogen and append with zeros if no repurpose investigation

In [56]:
# Example: Calling function with hydrogen_investment = False
df_edges_complete = expand_with_hydrogen(df_edges_cap_cost, hydrogen_investment)

In [57]:
# Extract country codes matching the pattern from Source
source_codes = df_edges_complete['Source'].str.extract(r'([A-Z]{2,3})_LNG_imp')[0].dropna().unique()

# Extract country codes matching the pattern from Destination
dest_codes = df_edges_complete['Destination'].str.extract(r'([A-Z]{2,3})_LNG_exp')[0].dropna().unique()

# Combine and remove duplicates
LNG_countries = set(source_codes) | set(dest_codes)

# Convert to sorted list
LNG_countries_list = sorted(LNG_countries)

In [58]:
#move to later step
df_network_nodes = extract_unique_nodes(df_edges_complete)

In [59]:
df_missing_supply_values_for_nodes  = create_missing_prod_and_lng_nodes(df_supply_demand, LNG_countries_list)

In [60]:
df_all_supply_demand = add_missing_supply_nodes(df_supply_demand, df_missing_supply_values_for_nodes)

In [61]:
# Example: Calling function with hydrogen_investment = False
df_demand_supply_complete = expand_with_hydrogen(df_all_supply_demand, hydrogen_investment)

# Display the result
df_demand_supply_complete

,Commodity,Node,Supply
0,Methane,AL,-9856.351937
1,Methane,AT,-67230.091737
2,Methane,BE,-133908.961538
3,Methane,BA,-5110.000000
4,Methane,BG,-24679.020000
...,...,...,...
347,Hydrogen,IN_LNG_imp,0.000000
348,Hydrogen,AS_LNG_imp,0.000000
349,Hydrogen,RU_LNG_imp,0.000000
350,Hydrogen,EG_LNG_imp,0.000000


In [62]:
#get all commodities of the model
df_commodities = extract_unique_commodities(df_demand_supply_complete)

#get separate df for the edges
df_edges_only = df_edges_cap_cost[['Source', 'Destination']]
#probably it is df_edges_cap_cost as now all edges are defined twice

### update relevant parameters for analysis

In [63]:
# Update the costs_edge parameter for Russia to Europe to avoid the use of Russian pipeline gas
df_edges_complete = update_parameter(df_edges_complete, df_updates, "costs_edge", "costs_new")
#df_edges_complete = update_parameter(df_edges_complete, df_updates, "max_capacities", "capacity_new")
#df_edges_complete = update_parameter(df_edges_complete, df_updates, "initial_capacities", "capacity_new")

In [64]:
if investment==True:
    #update investment cost values based on distances of connections
    df_updates_invest = update_investment_cost(df_updates_invest, df_distances)
    #only relevant in the invest case to update investment cost
    df_edges_complete = update_parameter(df_edges_complete, df_updates_invest, "new_build_cost", "costs_new")
    #only relevant in the invest case to update investment limit
    df_edges_complete = update_parameter(df_edges_complete, df_updates_invest , "max_capacities", "limit_new")

# export

In [65]:
# Export to Excel with multiple sheets
with pd.ExcelWriter(full_output_file_path, engine='openpyxl') as writer:
    df_network_nodes.to_excel(writer, index=False, sheet_name='Nodes')
    df_commodities.to_excel(writer, index=False, sheet_name='Commodities')
    df_edges_only.to_excel(writer, index=False, sheet_name='Edges')
    df_edges_complete.to_excel(writer, index=False, sheet_name='Parameters')
    df_demand_supply_complete.to_excel(writer, index=False, sheet_name='Supply')

In [33]:
# Export to Excel with multiple sheets
#with pd.ExcelWriter(full_edges_raw_file_path, engine='openpyxl') as writer:
#    df_edges_cap_cost.to_excel(writer, index=False, sheet_name='edges_raw')